# Exploration lane — 3 idea probes (Colab RTX PRO 6000, **NOT paper-grade**)

Cheap probes of 3 allocation/schedule ideas vs CARD-PB (FLOPs 0.665, l1 holds). **Exploration signal only** — Kiet reads `SUMMARY.md` and decides a proper port. Writes ONLY to Drive `results/exploration/`, tagged `exploration:true device:colab`. Pinned to `2a1f0e28`. Run All on an RTX PRO 6000.


## 1 · Setup — clone @ pinned commit, install

In [ ]:
!git clone --quiet https://github.com/anhkiet287/attackdro.git 2>/dev/null || (cd attackdro && git fetch --quiet origin)
%cd attackdro
!git checkout --quiet 2a1f0e28
!pip install --quiet -e . 2>/dev/null || true
import os; os.environ['PYTHONPATH']='/content/attackdro/src'
!git log --oneline -1

## 2 · Drive + guards (eps threat-model, device)

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, yaml, torch
EXPDIR='/content/drive/MyDrive/attackdro/results/exploration'
os.makedirs(EXPDIR, exist_ok=True)
tm=yaml.safe_load(open('configs/base.yaml'))['threat_model']
assert abs(tm['linf']['eps']-8/255)<1e-6 and tm['l2']['eps']==0.5 and tm['l1']['eps']==12, 'EPS GUARD FAILED'
print('eps guard OK:', tm)
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (WARN: expected RTX PRO 6000)')

## 3 · W&B (online, tags come from the configs)
Colab needs a one-time login. If you prefer no W&B, set `WANDB_MODE=disabled` below (reads still work — they come from the on-Drive `train.json`/`eval.json`).

In [ ]:
import os
os.environ['WANDB_MODE']='online'   # or 'disabled'
if os.environ['WANDB_MODE']=='online':
    try:
        import wandb; wandb.login()   # paste API key if prompted
    except Exception as e:
        print('wandb login skipped:', e); os.environ['WANDB_MODE']='disabled'

## 4 · Idea 1 — batch fail-rate threshold allocation (ep20)
Read: `exp/steps_{norm}` (did l1 self-select many steps, l∞ few?), `exp/attack_flops_ratio` vs 0.665, l1 train-vs-eval gap. **Kill:** l1 gap blows up OR FLOPs ≥ 0.665.

In [ ]:
!python scripts/train.py --config configs/exploration/idea1_failrate.yaml --seed 0 --run-name idea1_failrate --set results_dir={EXPDIR}/
!python scripts/evaluate.py --config configs/exploration/idea1_failrate.yaml --checkpoint {EXPDIR}/idea1_failrate/s0/ckpt/last.pt --run-name idea1_failrate --seed 0 --tier in-house -n 1000 --version apgd --set results_dir={EXPDIR}/

## 5 · Idea 2 — k_min self-recovery (kspan=4, k_min=1, ep30)
Read: `floor/l1` EVERY epoch (dip→climb = self-recovery?) + final l1 vs de-risk 45.6. **Kill:** floor stuck low + l1 tanks.

In [ ]:
!python scripts/train.py --config configs/exploration/idea2_kmin_recovery.yaml --seed 0 --run-name idea2_kmin_recovery --set results_dir={EXPDIR}/
!python scripts/evaluate.py --config configs/exploration/idea2_kmin_recovery.yaml --checkpoint {EXPDIR}/idea2_kmin_recovery/s0/ckpt/last.pt --run-name idea2_kmin_recovery --seed 0 --tier in-house -n 1000 --version apgd --set results_dir={EXPDIR}/

## 6 · Idea 3 — curriculum (3→6→10) vs fixed-10 control (ep30 each)
Read: does curriculum match the control's union/l1 at ep30 with lower cumulative FLOPs? **Kill:** curriculum l1 lags control OR no FLOPs saving.

In [ ]:
!python scripts/train.py --config configs/exploration/idea3_curriculum.yaml --seed 0 --run-name idea3_curriculum --set results_dir={EXPDIR}/
!python scripts/evaluate.py --config configs/exploration/idea3_curriculum.yaml --checkpoint {EXPDIR}/idea3_curriculum/s0/ckpt/last.pt --run-name idea3_curriculum --seed 0 --tier in-house -n 1000 --version apgd --set results_dir={EXPDIR}/
!python scripts/train.py --config configs/exploration/idea3_control_fixed10.yaml --seed 0 --run-name idea3_control_fixed10 --set results_dir={EXPDIR}/
!python scripts/evaluate.py --config configs/exploration/idea3_control_fixed10.yaml --checkpoint {EXPDIR}/idea3_control_fixed10/s0/ckpt/last.pt --run-name idea3_control_fixed10 --seed 0 --tier in-house -n 1000 --version apgd --set results_dir={EXPDIR}/

## 7 · Read → verdicts + SUMMARY

In [ ]:
!python scripts/dev/exploration_read.py {EXPDIR}
from IPython.display import Markdown, display
display(Markdown(open(f'{EXPDIR}/SUMMARY.md').read()))

---
**STOP here.** These are exploration signal, not paper numbers. Send `SUMMARY.md` (+ the per-idea `idea{1,2,3}_*.md`) to Kiet; he decides port / drop / different test. No proper ports from this notebook.